# Домашнє завдання. Тема 4
## Метрики відстані, матриці відстаней та кореляція

У роботі використано набір даних **Breast Cancer Wisconsin** з бібліотеки `sklearn`.

Мета: дослідити структуру даних, стандартизувати ознаки, побудувати візуалізації, обчислити матриці відстаней для різних метрик та зробити висновки щодо їх використання.

## 1. Імпорт бібліотек та завантаження даних

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

sns.set_theme(style="whitegrid")

In [ ]:
data = load_breast_cancer()

print(data.DESCR[:2500])

## 2. Створення DataFrame

In [ ]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

df.head()

## 3. Інформація про дані

In [ ]:
df.info()

## 4. Описові статистики

In [ ]:
df.describe().T

## 5. Стандартизація даних

Ознаки мають різні масштаби, тому перед обчисленням відстаней їх потрібно стандартизувати.
Стандартизація переводить кожну ознаку до середнього значення 0 та стандартного відхилення 1.

In [ ]:
features = df.drop(columns=["target", "target_name"])

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

df_scaled = pd.DataFrame(scaled_features, columns=data.feature_names)
df_scaled["target"] = df["target"]
df_scaled["target_name"] = df["target_name"]

df_scaled.head()

In [ ]:
df_scaled.describe().T.head(10)

## 6. Точкові діаграми

У наборі даних 30 ознак, тому `pairplot` для всіх стовпців буде дуже великим і важким для перегляду.
Для наочності використаємо частину найбільш зрозумілих ознак.

In [ ]:
selected_columns = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "target_name"
]

sns.pairplot(
    df[selected_columns],
    hue="target_name",
    diag_kind="hist",
    corner=True
)
plt.show()

## Додаткова кореляційна матриця

Кореляція дозволяє оцінити зв'язки між ознаками. Значення, близькі до 1 або -1, означають сильний зв'язок.

In [ ]:
plt.figure(figsize=(14, 10))
correlation_matrix = features.corr()
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation matrix of Breast Cancer features")
plt.show()

## Аналіз кореляції з цільовою змінною

Окремо переглянемо, які ознаки мають найсильніший зв'язок із класом пухлини (`target`). Це допомагає зрозуміти, які характеристики найбільше пов'язані з розділенням класів `malignant` та `benign`.

In [ ]:
target_correlation = df.drop(columns=["target_name"]).corr(numeric_only=True)["target"].sort_values(ascending=False)

target_correlation

In [ ]:
plt.figure(figsize=(8, 10))
target_correlation.drop("target").sort_values().plot(kind="barh")
plt.title("Feature correlation with target")
plt.xlabel("Correlation coefficient")
plt.ylabel("Feature")
plt.show()

## 7. Обчислення матриць відстаней

Для повного датасету матриця відстаней має розмір 569 × 569. Її можна обчислити повністю, але для зручної візуалізації heatmap використаємо перші 50 записів після стандартизації. Такий підхід дозволяє порівняти метрики без перевантаження графіків.

Метрики:
- `euclidean` — евклідова відстань;
- `cityblock` — мангеттенська відстань;
- `manhattan` — синонім `cityblock`;
- `l1` — також еквівалент мангеттенській відстані;
- `cosine` — косинусна відстань, яка оцінює напрям векторів.


In [ ]:
sample_size = 50
X_sample = df_scaled.drop(columns=["target", "target_name"]).iloc[:sample_size]

metrics = ["cityblock", "cosine", "euclidean", "l1", "manhattan"]

distance_matrices = {}

for metric in metrics:
    distance_matrices[metric] = cdist(X_sample, X_sample, metric=metric)
    print(f"{metric}: shape = {distance_matrices[metric].shape}")

### Додатково: розмір повної матриці відстаней

Для всього датасету можна обчислити повну матрицю 569 × 569. Нижче показано її розмір для евклідової метрики. Для heatmap далі використовується менша вибірка, щоб графіки залишались читабельними.

In [ ]:
X_full = df_scaled.drop(columns=["target", "target_name"])
full_euclidean_matrix = cdist(X_full, X_full, metric="euclidean")

full_euclidean_matrix.shape

## 8. Візуалізація матриць відстаней

In [ ]:
for metric, matrix in distance_matrices.items():
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, cmap="viridis")
    plt.title(f"Distance matrix: {metric}")
    plt.xlabel("Object index")
    plt.ylabel("Object index")
    plt.show()

## Порівняння середніх значень відстаней

In [ ]:
comparison = []

for metric, matrix in distance_matrices.items():
    upper_triangle = matrix[np.triu_indices_from(matrix, k=1)]
    comparison.append({
        "metric": metric,
        "mean_distance": upper_triangle.mean(),
        "min_distance": upper_triangle.min(),
        "max_distance": upper_triangle.max(),
        "std_distance": upper_triangle.std()
    })

comparison_df = pd.DataFrame(comparison).sort_values("mean_distance")
comparison_df

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=comparison_df, x="metric", y="mean_distance")
plt.title("Average distance by metric")
plt.xlabel("Metric")
plt.ylabel("Mean distance")
plt.show()

## 9. Висновки

1. Набір даних Breast Cancer містить 569 об'єктів та 30 числових ознак, які описують характеристики клітинних ядер. Цільова змінна має два класи: злоякісна (`malignant`) та доброякісна (`benign`) пухлина.

2. Ознаки мають різні масштаби, тому перед обчисленням відстаней була виконана стандартизація за допомогою `StandardScaler`. Це важливо, бо без стандартизації ознаки з великими числовими значеннями, наприклад площа, могли б домінувати над іншими ознаками.

3. Для `pairplot` було використано репрезентативну підмножину ознак, оскільки повний pairplot для 30 ознак створює понад 900 графіків і є важким для аналізу. Навіть на вибраних ознаках видно, що характеристики, пов'язані з радіусом, периметром і площею, допомагають розділяти класи.

4. Кореляційна матриця показала сильний зв'язок між деякими ознаками. Зокрема, радіус, периметр і площа мають високу позитивну кореляцію між собою, що логічно, оскільки всі вони описують розмір клітинного ядра.

5. Аналіз кореляції з цільовою змінною показує, які ознаки найбільше пов'язані з класом пухлини. Це корисно для попереднього аналізу даних перед побудовою моделей машинного навчання.

6. Метрики `cityblock`, `manhattan` та `l1` дали однакові результати, оскільки всі вони відповідають L1-відстані. Тому в практичному аналізі достатньо використовувати одну з них.

7. Евклідова відстань (`euclidean`) добре підходить для стандартизованих числових даних, оскільки вимірює геометричну близькість об'єктів у багатовимірному просторі. Саме її часто використовують як базову метрику в задачах кластеризації.

8. Мангеттенська відстань (`manhattan`, `cityblock`, `l1`) може бути корисною, коли важливо враховувати сумарні абсолютні відхилення за всіма ознаками. Вона може бути менш чутливою до окремих великих відхилень, ніж евклідова метрика.

9. Косинусна відстань (`cosine`) оцінює не абсолютну різницю між значеннями, а подібність напрямків векторів. Вона може бути корисною, коли важливіше співвідношення між ознаками, а не абсолютна величина значень.

10. Для цього набору даних найбільш практичними є `euclidean` та `manhattan` / `cityblock` метрики. Евклідова метрика добре відображає загальну геометричну близькість об'єктів, а мангеттенська є хорошою альтернативою для перевірки стабільності результатів.

Загалом, вибір метрики залежить від задачі. Для кластеризації стандартизованих числових медичних даних доцільно починати з евклідової відстані, а потім порівнювати результати з мангеттенською та косинусною метриками.


## Коментар для LMS

Обраний підхід до виконання ДЗ: намагатися отримати вищий бал шляхом можливого подальшого доопрацювання роботи відповідно до фідбеку ментора.
